# Game Intelligence AI — Stage 03: QLoRA Training Readiness

هدف هذا الدفتر فحص بيئة Google Colab والتحقق عمليًا من إمكانية تحميل نموذج 3B بتقنية 4-bit QLoRA، وتنفيذ خطوة forward/backward، ثم حفظ LoRA adapter وإعادة تحميله.

- لا يستخدم Google Drive.
- لا يبدأ تدريبًا كاملًا.
- تخزين `/content` مؤقت؛ نزّل التقرير والـadapter قبل انتهاء الجلسة.
- لا ترفع الأوزان أو البيانات أو مفاتيح الوصول إلى GitHub.
- نجاح الاختبار القصير لا يضمن نجاح تدريب طويل.

النموذج الأساسي: `Qwen/Qwen2.5-Coder-3B-Instruct`


## 0. الإعداد — شغّل الخلايا بالترتيب

In [ ]:
import os, sys, json, time, platform, subprocess, gc
from pathlib import Path
from datetime import datetime, timezone

BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODEL_7B = "Qwen/Qwen2.5-Coder-7B-Instruct"
RUN_7B_PROBE = True
MAX_LENGTH = 512

ROOT = Path("/content/game_intelligence_ai_stage3")
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
REPORT_PATH = Path("/content/stage3_qlora_readiness_report.json")

report = {
    "project": "Game Intelligence AI", "stage": 3,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL, "checks": {}, "errors": [], "readiness": False
}
print("Python:", sys.version)
print("Platform:", platform.platform())


## 1. تشخيص GPU وVRAM والقرص وPyTorch — دون تغيير الإصدارات

In [ ]:
def cmd(args):
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=30)
        return {"code": p.returncode, "stdout": p.stdout.strip(), "stderr": p.stderr.strip()}
    except Exception as e:
        return {"code": -1, "error": repr(e)}

gpu = cmd(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"])
disk = cmd(["bash", "-lc", "df -h /content | tail -1"])
try:
    import torch
    ti = {"version": torch.__version__, "cuda": torch.cuda.is_available(),
          "torch_cuda": torch.version.cuda, "devices": []}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            ti["devices"].append({"name": p.name, "vram_gib": round(p.total_memory/1024**3,2),
                                  "compute": f"{p.major}.{p.minor}"})
except Exception as e:
    ti = {"error": repr(e)}
report["environment"] = {"gpu": gpu, "disk": disk, "torch": ti}
print(json.dumps(report["environment"], indent=2))


## 2. تثبيت المكتبات المطلوبة دون إعادة تثبيت PyTorch أو CUDA

In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets


In [ ]:
import importlib.metadata as md
versions = {}
for p in ["torch","transformers","accelerate","peft","bitsandbytes","datasets"]:
    try: versions[p] = md.version(p)
    except Exception as e: versions[p] = "NOT INSTALLED: " + str(e)
report["package_versions"] = versions
print(json.dumps(versions, indent=2))
print("إذا طلب Colab إعادة تشغيل الجلسة: أعد التشغيل ثم شغّل الدفتر من البداية.")


## 3. اختبار استيراد BitsAndBytes والمكتبات

In [ ]:
bnb_status = {"imports_ok": False, "cuda_4bit_load_ok": False}
try:
    import torch, bitsandbytes, transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    bnb_status.update({"imports_ok": True,
                       "bitsandbytes_version": getattr(bitsandbytes,"__version__","unknown"),
                       "transformers_version": transformers.__version__})
except Exception as e:
    bnb_status["import_error"] = repr(e)
    report["errors"].append({"step":"imports","error":repr(e)})
report["checks"]["bitsandbytes"] = bnb_status
print(json.dumps(bnb_status, indent=2))


## 4. تحميل نموذج 3B فعليًا بكمية 4-bit NF4
لا يُحسب fallback إلى FP16 على أنه نجاح QLoRA.

In [ ]:
model = tokenizer = None
t0 = time.time()
try:
    assert torch.cuda.is_available(), "CUDA GPU غير متاح."
    assert bnb_status["imports_ok"], "فشل استيراد المكتبات."
    qconfig = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=qconfig, device_map="auto",
        torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    model.config.use_cache = False
    bnb_status.update({"cuda_4bit_load_ok":True, "load_seconds":round(time.time()-t0,2),
                       "device_map":str(getattr(model,"hf_device_map",{}))})
except Exception as e:
    bnb_status["load_error"] = repr(e)
    report["errors"].append({"step":"4bit_model_load","error":repr(e)})
    model = tokenizer = None
report["checks"]["bitsandbytes"] = bnb_status
print(json.dumps(bnb_status, indent=2))


## 5. اختبار QLoRA: forward/backward وفحص loss والـgradients

In [ ]:
smoke = {"attempted":False, "passed":False}
if model is not None and tokenizer is not None:
    try:
        smoke["attempted"] = True
        model = prepare_model_for_kbit_training(model)
        lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                          task_type="CAUSAL_LM",
                          target_modules=["q_proj","k_proj","v_proj","o_proj"])
        model = get_peft_model(model, lora)
        model.train()
        sample = ("### Task: Write a simple game update function.\n"
                  "### Answer:\nfunction update(player, dt) { player.x += player.vx * dt; }")
        batch = tokenizer(sample, return_tensors="pt", truncation=True,
                          max_length=MAX_LENGTH, padding="max_length")
        device = model.get_input_embeddings().weight.device
        batch = {k:v.to(device) for k,v in batch.items()}
        batch["labels"] = batch["input_ids"].clone()
        model.zero_grad(set_to_none=True)
        out = model(**batch)
        assert torch.isfinite(out.loss).item(), "Loss غير finite."
        out.loss.backward()
        trainable = [p for p in model.parameters() if p.requires_grad]
        grads = [p.grad for p in trainable if p.grad is not None]
        assert trainable, "لم يتم العثور على معاملات LoRA قابلة للتدريب."
        assert grads, "لم يتم إنتاج gradients."
        assert all(torch.isfinite(g).all().item() for g in grads), "Gradient غير finite."
        smoke.update({"passed":True, "loss":float(out.loss.detach().cpu()),
                      "trainable_tensors":len(trainable), "gradient_tensors":len(grads)})
        model.zero_grad(set_to_none=True)
    except Exception as e:
        smoke["error"] = repr(e)
        report["errors"].append({"step":"qlora_forward_backward","error":repr(e)})
else:
    smoke["skipped_reason"] = "لم ينجح تحميل نموذج 3B بكمية 4-bit."
report["checks"]["qlora_smoke_test"] = smoke
print(json.dumps(smoke, indent=2))


## 6. حفظ LoRA adapter ثم إعادة تحميله

In [ ]:
adapter_dir = ARTIFACTS / "qlora_adapter_smoke_test"
adapter = {"save_ok":False, "reload_ok":False}
if model is not None and smoke.get("passed"):
    try:
        model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
        adapter["save_ok"] = True
        adapter["path"] = str(adapter_dir)
        base = model.get_base_model()
        reloaded = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
        adapter["reload_ok"] = True
        del reloaded
    except Exception as e:
        adapter["error"] = repr(e)
        report["errors"].append({"step":"adapter_save_reload","error":repr(e)})
else:
    adapter["skipped_reason"] = "اختبار QLoRA لم ينجح."
report["checks"]["adapter_save_reload"] = adapter
print(json.dumps(adapter, indent=2))


## 7. اختبار اختياري لتحميل 7B فقط
لا يعمل إلا عند نجاح اختبارات 3B ووجود VRAM لا تقل عن 16 GiB. هذا اختبار تحميل فقط وليس تدريب 7B.

In [ ]:
probe7 = {"attempted":False, "passed":False}
vram = 0.0
if torch.cuda.is_available():
    vram = max(torch.cuda.get_device_properties(i).total_memory
               for i in range(torch.cuda.device_count())) / 1024**3
if RUN_7B_PROBE and smoke.get("passed") and adapter.get("reload_ok") and vram >= 16:
    probe7["attempted"] = True
    m7 = None
    try:
        q7 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True,
                                bnb_4bit_compute_dtype=torch.float16)
        t0 = time.time()
        m7 = AutoModelForCausalLM.from_pretrained(
            MODEL_7B, quantization_config=q7, device_map="auto",
            torch_dtype=torch.float16, low_cpu_mem_usage=True)
        probe7.update({"passed":True,"load_seconds":round(time.time()-t0,2)})
    except Exception as e:
        probe7["error"] = repr(e)
        report["errors"].append({"step":"7b_load_probe","error":repr(e)})
    finally:
        if m7 is not None: del m7
        gc.collect()
        torch.cuda.empty_cache()
else:
    probe7["skipped_reason"] = "تم تعطيله أو فشلت المتطلبات أو VRAM أقل من 16 GiB."
report["checks"]["optional_7b_load_probe"] = probe7
print(json.dumps(probe7, indent=2))


## 8. قرار الجاهزية النهائي وإنشاء تقرير JSON

In [ ]:
try:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        report["measurements"] = {
            "peak_allocated_gib":round(torch.cuda.max_memory_allocated()/1024**3,3),
            "peak_reserved_gib":round(torch.cuda.max_memory_reserved()/1024**3,3),
            "current_allocated_gib":round(torch.cuda.memory_allocated()/1024**3,3)
        }
except Exception as e:
    report["measurement_error"] = repr(e)

report["readiness"] = bool(
    bnb_status.get("cuda_4bit_load_ok") and smoke.get("passed")
    and adapter.get("save_ok") and adapter.get("reload_ok")
)
report["decision"] = ("READY_FOR_SEPARATE_SMALL_DATASET_TRIAL" if report["readiness"]
                      else "NOT_READY_FIX_RECORDED_ERRORS")
report["limitations"] = [
    "Smoke test only; not full fine-tuning.",
    "Passing does not guarantee long-run stability or model quality.",
    "Colab /content storage is temporary.",
    "Optional 7B probe is load-only."
]
REPORT_PATH.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps({"readiness":report["readiness"],"decision":report["decision"],
                  "report_path":str(REPORT_PATH),"checks":report["checks"],
                  "errors":report["errors"]},indent=2,ensure_ascii=False))


## 9. تنزيل التقرير والـadapter الاختياري

In [ ]:
from google.colab import files
files.download(str(REPORT_PATH))


In [ ]:
if adapter.get("save_ok"):
    import shutil
    archive = shutil.make_archive("/content/qlora_adapter_smoke_test","zip",
                                  root_dir=str(adapter_dir))
    print("Adapter archive:", archive)
    files.download(archive)
else:
    print("لم يتم إنشاء ZIP لأن حفظ adapter لم ينجح.")


## 10. ما الذي ترسله للمراجعة؟

1. شغّل جميع الخلايا بالترتيب.
2. نزّل `stage3_qlora_readiness_report.json`.
3. أرسل ملف JSON هنا.
4. إذا نجح حفظ adapter، يمكنك أيضًا إرسال ZIP عند الحاجة.

**لا نبدأ المرحلة الرابعة قبل مراجعة التقرير.** إذا كانت `readiness` تساوي `false`، نعالج الخطأ المسجل أولًا.